In [ ]:
import shutil
import os

# 在 Colab 的存储里创建文件夹（/content/ 是 Colab 的根目录）
extract_path = '/content/drive/MyDrive/Colab/urban2vec_complaint/data/raw'
os.makedirs(extract_path, exist_ok=True)

print(f"准备解压到：{extract_path}")

zip_path = '/content/drive/MyDrive/Colab/image_example.zip'  # 修改为你的压缩包所在路径

# 解压
shutil.unpack_archive(zip_path, extract_path)
print("✅ ZIP 解压完成！")

In [ ]:
# 1. 先确保你在正确的目录（如果 requirements.txt 在 Drive 里）
%cd /content/drive/MyDrive/Colab/urban2vec_complaint

# 2. 一键安装（最常用）
%pip install -r requirements.txt

#如果安装慢，用清华镜像加速（国内用户）
#%pip install -r requirements.txt -i https://pypi.tuna.tsinghua.edu.cn/simple

In [ ]:
%%bash
# 先检查 /content/ 下是否有社区街景文件夹
ls /content/drive/MyDrive/Colab/urban2vec_complaint/data/raw/image_example
# 应该看到：左家庄街道_images/  杨宋镇_images/  等

军庄镇_images
北京雁栖经济开发区_images
北新桥街道_images
北臧村镇_images
卢沟桥街道_images
呼家楼街道_images
大孙各庄镇_images
广安门外街道_images
延庆镇_images
来广营地区_images


In [12]:
# Stage 0: 数据准备（生成元数据）
!python 0_adapt_data.py \
  --image_root "./data/raw/images/" \
  --output "./data/processed/metadata.csv"

# Stage 1: 构建训练对（生成 pickle 文件）
!python 1_build_pairs.py \
  --metadata "./data/processed/metadata.csv" \
  --output_dir "./data/processed/"

2026-02-07 09:48:52,125 - INFO - 开始处理图片目录: image_example
2026-02-07 09:48:52,126 - INFO - 随机种子: 42
2026-02-07 09:48:52,127 - INFO - 处理社区: 军庄镇
2026-02-07 09:48:52,152 - INFO -   找到 200 张有效图片
2026-02-07 09:48:52,152 - INFO - 处理社区: 北京雁栖经济开发区
2026-02-07 09:48:52,180 - INFO -   找到 200 张有效图片
2026-02-07 09:48:52,181 - INFO - 处理社区: 北新桥街道
2026-02-07 09:48:52,198 - INFO -   找到 200 张有效图片
2026-02-07 09:48:52,199 - INFO - 处理社区: 北臧村镇
2026-02-07 09:48:52,218 - INFO -   找到 200 张有效图片
2026-02-07 09:48:52,218 - INFO - 处理社区: 卢沟桥街道
2026-02-07 09:48:52,242 - INFO -   找到 200 张有效图片
2026-02-07 09:48:52,243 - INFO - 处理社区: 呼家楼街道
2026-02-07 09:48:52,269 - INFO -   找到 200 张有效图片
2026-02-07 09:48:52,270 - INFO - 处理社区: 大孙各庄镇
2026-02-07 09:48:52,287 - INFO -   找到 200 张有效图片
2026-02-07 09:48:52,288 - INFO - 处理社区: 广安门外街道
2026-02-07 09:48:52,306 - INFO -   找到 200 张有效图片
2026-02-07 09:48:52,306 - INFO - 处理社区: 延庆镇
2026-02-07 09:48:52,323 - INFO -   找到 200 张有效图片
2026-02-07 09:48:52,324 - INFO - 处理社区: 来广营地区
2026-02-07 09:48:52

In [6]:
# Stage 2: 训练街景嵌入模型（生成 .tar 模型文件）
!python 2_train_stage1.py \
  --train_pairs "./data/processed/train_pair_knn.pickle" \
  --val_pairs "./data/processed/val_pair_knn.pickle" \
  --image_root "./data/raw/images/" \
  --output_dir "./models/checkpoints/"

使用GPU: Tesla T4
2026-02-07 10:14:13,744 - INFO - 使用设备: cuda:0
2026-02-07 10:14:13,744 - INFO - 随机种子: 42
2026-02-07 10:14:13,744 - INFO - 加载训练对...
2026-02-07 10:14:14,744 - INFO - 加载训练对: 9600 个
2026-02-07 10:14:14,744 - INFO - 加载验证对: 2400 个
2026-02-07 10:14:14,744 - INFO - 创建数据加载器...
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:627: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  warnings.warn(
2026-02-07 10:14:14,748 - INFO - 训练集大小: 9600，批大小: 32
2026-02-07 10:14:14,748 - INFO - 验证集大小: 2400，批大小: 32
2026-02-07 10:14:14,748 - INFO - 创建模型，嵌入维度: 50
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretra

In [7]:
# Stage 3: 提取社区嵌入向量（使用 Stage 2 的模型）
!python 3_extract_embedding.py \
  --image_root "./data/raw/images/" \
  --model_path "./models/checkpoints/street_view_best.tar" \
  --output "./embeddings/train/community_embeddings.csv"

使用GPU: Tesla T4
2026-02-07 12:50:14,080 - INFO - 使用设备: cuda:0
2026-02-07 12:50:14,081 - INFO - 扫描图片目录: ./image_example
2026-02-07 12:50:14,082 - INFO - 扫描社区: 军庄镇
2026-02-07 12:50:14,096 - INFO -   找到 200 张有效图片
2026-02-07 12:50:14,096 - INFO - 扫描社区: 北京雁栖经济开发区
2026-02-07 12:50:14,113 - INFO -   找到 200 张有效图片
2026-02-07 12:50:14,113 - INFO - 扫描社区: 北新桥街道
2026-02-07 12:50:14,130 - INFO -   找到 200 张有效图片
2026-02-07 12:50:14,130 - INFO - 扫描社区: 北臧村镇
2026-02-07 12:50:14,147 - INFO -   找到 200 张有效图片
2026-02-07 12:50:14,148 - INFO - 扫描社区: 卢沟桥街道
2026-02-07 12:50:14,161 - INFO -   找到 200 张有效图片
2026-02-07 12:50:14,161 - INFO - 扫描社区: 呼家楼街道
2026-02-07 12:50:14,172 - INFO -   找到 200 张有效图片
2026-02-07 12:50:14,172 - INFO - 扫描社区: 大孙各庄镇
2026-02-07 12:50:14,185 - INFO -   找到 200 张有效图片
2026-02-07 12:50:14,185 - INFO - 扫描社区: 广安门外街道
2026-02-07 12:50:14,195 - INFO -   找到 200 张有效图片
2026-02-07 12:50:14,196 - INFO - 扫描社区: 延庆镇
2026-02-07 12:50:14,209 - INFO -   找到 200 张有效图片
2026-02-07 12:50:14,209 - INFO - 扫描社区: 来广营地区

In [10]:
# Stage 4: 训练投诉预测模型
!python 4_train_predictor.py \
  --embeddings "./embeddings/train/community_embeddings.csv" \
  --complaints "./data/raw/complaints.csv" \
  --output_dir "./results/"

2026-02-07 12:56:53,326 - INFO - 开始投诉预测模型训练...
2026-02-07 12:56:53,326 - INFO - 嵌入向量: community_embeddings.csv
2026-02-07 12:56:53,326 - INFO - 投诉数据: complaints_example.csv
2026-02-07 12:56:53,326 - INFO - 输出目录: models
2026-02-07 12:56:53,326 - INFO - 加载嵌入向量: community_embeddings.csv
2026-02-07 12:56:53,332 - INFO - 加载投诉数据: complaints_example.csv
2026-02-07 12:56:53,337 - INFO - 合并后数据: 10 个样本，59 个特征
2026-02-07 12:56:53,339 - INFO - 特征维度: (10, 50), 目标维度: (10, 8)
2026-02-07 12:56:53,339 - INFO - 可用目标: ['total_complaints', 'house_management', 'traffic_municipal', 'public_service', 'social_affairs', 'market_economy', 'urban_rural', 'OTHER']
2026-02-07 12:56:53,339 - INFO - 
训练模型: Ridge
2026-02-07 12:56:53,339 - INFO - 评估模型 Ridge 对目标 total_complaints
2026-02-07 12:56:53,364 - INFO -   total_complaints - R²: -1.7499 ± 0.1050, MAE: 179.4559 ± 1.7369
2026-02-07 12:56:53,364 - INFO - 评估模型 Ridge 对目标 house_management
2026-02-07 12:56:53,370 - INFO -   house_management - R²: -3.7257 ± 1.0435, MAE: